Cleaning data, choose companies that have layoff from 1.10.2025

In [ ]:
import pandas as pd
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from config.config import GET_FINANCIAL_DATA

path = GET_FINANCIAL_DATA["INPUT_CSV_PATH"]
print(f"Reading data from: {path}")
df = pd.read_csv(GET_FINANCIAL_DATA["INPUT_CSV_PATH"])

# # Filter out companies from 10.2025
df['Date'] = pd.to_datetime(df['Date'], format='%d.%m.%Y')
df = df[df['Date'] >= '2025-10-01']

df.to_csv(GET_FINANCIAL_DATA["FILTERED_CSV_PATH"], index=False)


Reading data from: c:\Users\phuon\Desktop\osint-proj\layoff-detector\data\raw\fyi_layoffs.csv


Get tickers of companies and appends to layoffs.csv

In [3]:
import yfinance as yf
import pandas as pd

df = pd.read_csv(GET_FINANCIAL_DATA["FILTERED_CSV_PATH"])

tickers = []

for company in df['Company']:
    results = yf.Search(company).quotes
    if results:
        search_result = results[0]
        tickers.append(search_result.get('symbol'))
    else:
        tickers.append(None)

df['Ticker'] = tickers

df.to_csv(GET_FINANCIAL_DATA["LAYOFFS_WITH_TICKERS_CSV_PATH"], index=False)

Cleaning data, remove companies without tickers

In [4]:
import pandas as pd
df = pd.read_csv(GET_FINANCIAL_DATA["LAYOFFS_WITH_TICKERS_CSV_PATH"])
df.dropna(subset=['Ticker'], inplace=True)
df.to_csv(GET_FINANCIAL_DATA["LAYOFFS_WITH_TICKERS_CSV_PATH"], index=False)

Get quarterly balance sheet of companies

In [ ]:
import yfinance as yf
import pandas as pd

df = pd.read_csv(GET_FINANCIAL_DATA["LAYOFFS_WITH_TICKERS_CSV_PATH"])
tickers = df['Ticker'].tolist()
for ticker in tickers:
    try:
        cashflow = yf.Ticker(ticker).quarterly_balance_sheet
        print(f"Fetched balance sheet data for {ticker}")
        cashflow.to_csv(GET_FINANCIAL_DATA["BALANCE_SHEET_DIR"]/ f"{ticker}_balancesheet.csv")
    except Exception as e:
        print(f"Error fetching balance sheet data for {ticker}: {e}")

Get quarterly cashflow of companies

In [ ]:
import yfinance as yf
import pandas as pd

df = pd.read_csv(GET_FINANCIAL_DATA["LAYOFFS_WITH_TICKERS_CSV_PATH"])
tickers = df['Ticker'].tolist()
for ticker in tickers:
    try:
        cashflow = yf.Ticker(ticker).quarterly_cashflow
        print(f"Fetched cashflow data for {ticker}")
        cashflow.to_csv(GET_FINANCIAL_DATA["CASHFLOW_DIR"] / f"{ticker}_cashflow.csv")
    except Exception as e:
        print(f"Error fetching cashflow data for {ticker}: {e}")

Get financial data for each ticker and save to CSV files.

In [ ]:
import yfinance as yf
import pandas as pd
df = pd.read_csv(GET_FINANCIAL_DATA["LAYOFFS_WITH_TICKERS_CSV_PATH"])
financials = []

for ticker in df['Ticker']:
    try:
        financial_data = yf.Ticker(ticker).quarterly_financials
        financial_data.to_csv(GET_FINANCIAL_DATA["FINANCIAL_DATA_DIR"]/ f"{ticker}_financials.csv")
        print(f"Fetched financial data for {ticker}")
    except Exception as e:
        print(f"Error fetching financial data for {ticker}: {e}")